In [18]:
import os
import time
import subprocess

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor, GBTRegressor

import os

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

if 'JAVA_TOOL_OPTIONS' in os.environ:
    del os.environ['JAVA_TOOL_OPTIONS']

print("JAVA_HOME:", os.environ['JAVA_HOME'])


spark = SparkSession.builder \
    .appName("ModeleImmoRhone") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [19]:
bases_fusionnees = spark.read.csv("bases_fusionnees.csv", header=True, inferSchema=True, sep=",")

In [20]:
bases_fusionnees.columns

['type_bien', 'pieces', 'prix', 'surface', 'etage', 'ville', 'code_postal']

In [21]:
print("=== Statistiques descriptives ===")
bases_fusionnees.describe().show()

print("=== Repartition par type de bien ===")
bases_fusionnees.groupBy("type_bien").count().orderBy(F.desc("count")).show()

print("=== Repartition par nombre de pieces ===")
bases_fusionnees.groupBy("pieces").count().orderBy("pieces").show()

print("=== Top 10 villes ===")
bases_fusionnees.groupBy("ville").count().orderBy(F.desc("count")).show(10)

=== Statistiques descriptives ===
+-------+-----------+------------------+-----------------+------------------+------------------+------+------------------+
|summary|  type_bien|            pieces|             prix|           surface|             etage| ville|       code_postal|
+-------+-----------+------------------+-----------------+------------------+------------------+------+------------------+
|  count|       8019|              8019|             8019|              8019|              8019|  8019|              8019|
|   mean|       NULL|  4.05100386581868|427315.1996508293|101.58756702830776|2.8797779201366644|  NULL|  69273.8709315376|
| stddev|       NULL|1.6509496507084573|300296.6495465243| 61.13174260486775|  2.33179621631308|  NULL|252.03671516019935|
|    min|appartement|               1.0|          49000.0|               9.0|                -1|affoux|             69001|
|    max|     maison|              12.0|        4160000.0|             900.0|               RDC| éveux|  

In [22]:
# Encodage des colonnes categoriques
indexer_ville = StringIndexer(inputCol="ville",       outputCol="ville_index",  handleInvalid="keep")
indexer_type  = StringIndexer(inputCol="type_clean",  outputCol="type_index",   handleInvalid="keep")
indexer_etage = StringIndexer(inputCol="etage_clean", outputCol="etage_index",  handleInvalid="keep")

# Liste des features
features_cols = [
    "surface", "pieces", "chambres", "terrain", "neuf_flag",
    "ville_index", "type_index", "etage_index"
]

assembler = VectorAssembler(inputCols=features_cols, outputCol="features")

print("Features utilisees :")
for f in features_cols:
    print(f"  - {f}")

Features utilisees :
  - surface
  - pieces
  - chambres
  - terrain
  - neuf_flag
  - ville_index
  - type_index
  - etage_index


In [23]:
train_data, test_data = bases_fusionnees.randomSplit([0.8, 0.2], seed=42)

train_data = train_data.cache()
test_data  = test_data.cache()

print(f"Taille train : {train_data.count()} lignes")
print(f"Taille test  : {test_data.count()} lignes")

Taille train : 6486 lignes
Taille test  : 1533 lignes


In [24]:
bases_fusionnees.printSchema()
print(bases_fusionnees.columns)

root
 |-- type_bien: string (nullable = true)
 |-- pieces: double (nullable = true)
 |-- prix: double (nullable = true)
 |-- surface: double (nullable = true)
 |-- etage: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: integer (nullable = true)

['type_bien', 'pieces', 'prix', 'surface', 'etage', 'ville', 'code_postal']


In [25]:
train_data, test_data = bases_fusionnees.randomSplit([0.8, 0.2], seed=42)

In [26]:
train_data

DataFrame[type_bien: string, pieces: double, prix: double, surface: double, etage: string, ville: string, code_postal: int]

In [27]:
indexer_ville = StringIndexer(inputCol="ville", outputCol="ville_idx", handleInvalid="keep")
indexer_type  = StringIndexer(inputCol="type_bien", outputCol="type_idx", handleInvalid="keep")
indexer_etage = StringIndexer(inputCol="etage", outputCol="etage_idx", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["type_idx", "pieces", "surface", "etage_idx", "ville_idx"],
    outputCol="features"
)

In [28]:
dt = DecisionTreeRegressor(featuresCol="features", labelCol="prix", maxDepth=8,maxBins=256, seed=42)
pipeline_dt = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, dt])

t0 = time.perf_counter()
model_dt = pipeline_dt.fit(train_data)
dt_train_time = time.perf_counter() - t0
print(f"Decision Tree - temps entrainement : {dt_train_time:.2f} s")

Decision Tree - temps entrainement : 1.83 s


In [29]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="prix",
    maxBins=256,
    numTrees=100,
    maxDepth=10,
    seed=42
)

pipeline_rf = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, rf])

t0 = time.perf_counter()
model_rf = pipeline_rf.fit(train_data)
rf_train_time = time.perf_counter() - t0

print(f"Random Forest - temps entrainement : {rf_train_time:.2f} s")

Random Forest - temps entrainement : 17.70 s


In [30]:
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="prix",
    maxIter=100,
    maxDepth=8,
    maxBins=256,
    stepSize=0.1,
    seed=42
)

pipeline_gbt = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, gbt])

t0 = time.perf_counter()
model_gbt = pipeline_gbt.fit(train_data)
gbt_train_time = time.perf_counter() - t0

print(f"GBT/XGBoost - temps entrainement : {gbt_train_time:.2f} s")

GBT/XGBoost - temps entrainement : 110.14 s


In [31]:
# Prédictions sur le jeu de test
dt_pred  = model_dt.transform(test_data)
rf_pred  = model_rf.transform(test_data)
gbt_pred = model_gbt.transform(test_data)

# Evaluateurs
r2_eval   = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="r2")
rmse_eval = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="rmse")
mae_eval  = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="mae")

# Calcul des métriques
dt_r2,  dt_rmse,  dt_mae  = r2_eval.evaluate(dt_pred),  rmse_eval.evaluate(dt_pred),  mae_eval.evaluate(dt_pred)
rf_r2,  rf_rmse,  rf_mae  = r2_eval.evaluate(rf_pred),  rmse_eval.evaluate(rf_pred),  mae_eval.evaluate(rf_pred)
gbt_r2, gbt_rmse, gbt_mae = r2_eval.evaluate(gbt_pred), rmse_eval.evaluate(gbt_pred), mae_eval.evaluate(gbt_pred)

# Tableau comparatif
sep = "=" * 72
print(sep)
print(f"{'Metrique':<20} {'Decision Tree':>15} {'Random Forest':>15} {'GBT/XGBoost':>15}")
print(sep)
print(f"{'R2':<20} {dt_r2:>15.4f} {rf_r2:>15.4f} {gbt_r2:>15.4f}")
print(f"{'RMSE (euros)':<20} {dt_rmse:>15,.0f} {rf_rmse:>15,.0f} {gbt_rmse:>15,.0f}")
print(f"{'MAE (euros)':<20} {dt_mae:>15,.0f} {rf_mae:>15,.0f} {gbt_mae:>15,.0f}")
print(f"{'Temps train (s)':<20} {dt_train_time:>15.2f} {rf_train_time:>15.2f} {gbt_train_time:>15.2f}")
print(sep)

scores = {"Decision Tree": dt_r2, "Random Forest": rf_r2, "GBT/XGBoost": gbt_r2}
best   = max(scores, key=scores.get)
print(f"\n=> Meilleur modele : {best} (R2 = {scores[best]:.4f})")

Metrique               Decision Tree   Random Forest     GBT/XGBoost
R2                            0.5761          0.6408          0.5112
RMSE (euros)                 200,175         184,267         214,940
MAE (euros)                  105,386         106,517         113,177
Temps train (s)                 1.83           17.70          110.14

=> Meilleur modele : Random Forest (R2 = 0.6408)
